1. Importing Necessary Libraries

In [16]:
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
print('pandas', pd.__version__)

pandas 2.2.3


2. Configuration 

In [17]:
DATA_ROOT = '.'  
ANNOTATION_FILES = [
    'fruit_quality_split_annotations.csv',
    #'fruit_quality_adv_annotations/Annotations_ANN_fgsm.csv',
    'fruit_quality_adv_annotations/Annotations_ANN_pgd.csv',
    #'fruit_quality_adv_annotations/Annotations_CNN_fgsm.csv',
    'fruit_quality_adv_annotations/Annotations_CNN_pgd.csv',
    #'fruit_quality_adv_annotations/Annotations_RNN_fgsm.csv',
    #'fruit_quality_adv_annotations/Annotations_RNN_pgd.csv',
]
#OUT_FILENAME = 'fruit_quality_train_detector_annotations.csv'  
#OUT_FILENAME = 'fruit_quality_unkonwn_adversarial_fgsm_RNN_detector_annotations.csv'  
#OUT_FILENAME = 'fruit_quality_unkonwn_adversarial_PGD_detector_annotations.csv'  
OUT_FILENAME = 'fruit_quality_train_semisupervised_detector_annotations.csv'  
SEED = 42

3. Robust CSV reader, auto-detect delimiter and parse annotation files

In [18]:
POSSIBLE_DELIMS = [';', ',', '\t']

def detect_delim_firstline(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        first = f.readline()
    for d in POSSIBLE_DELIMS:
        if d == '\t':
            if '\t' in first:
                return '\t'
        else:
            if d in first:
                return d
    return ','

def read_annotation_csv(path):
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Annotation file not found: {path}')
    
    delim = detect_delim_firstline(path)
    sep = '\t' if delim == '\t' else delim
    try:
        df = pd.read_csv(path, sep=sep, engine='python', dtype=str)
    except Exception as e:
        lines = open(path, 'r', encoding='utf-8', errors='ignore').read().splitlines()
        rows = [l.split(';') for l in lines if l.strip()]
        df = pd.DataFrame(rows)
    df.columns = [c.strip() for c in df.columns]

    if len(df.columns) == 1:
        col0 = df.columns[0]
        if ';' in col0:
            new_cols = [c.strip() for c in col0.split(';')]
            split_rows = df[col0].str.split(';', expand=True)
            split_rows.columns = new_cols[:split_rows.shape[1]]
            df = split_rows
        else:
            split_rows = df[col0].str.split(';', expand=True)
            df = split_rows
    return df

4. Read & normalize all annotation files — merge into a single DataFrame

In [19]:
all_dfs = []
for ann in ANNOTATION_FILES:
    ann_path = os.path.join(DATA_ROOT, ann) if not os.path.isabs(ann) and os.path.exists(os.path.join(DATA_ROOT, ann)) else ann
    print('\nReading', ann_path)
    try:
        df = read_annotation_csv(ann_path)
    except Exception as e:
        print('ERROR reading', ann_path, e)
        continue

    source = Path(ann).stem
    df = df.copy()   

    img_col = None
    for cand in ['image_id','image','filename','file','path','image_path','img_path','Filepath','fileName','ImageId']:
        if cand in df.columns:
            img_col = cand; break
    if img_col is None:
        img_col = df.columns[0]
        print(f'Using first column as image column for {ann}:', img_col)

    df = df.rename(columns={img_col: 'image_id'})
    
    for cand in ['label','class','orig_class','y','target']:
        if cand in df.columns:
            df = df.rename(columns={cand:'label'}); break
    for cand in ['isNight','is_night','night']:
        if cand in df.columns:
            df = df.rename(columns={cand:'isNight'}); break
    for cand in ['split','set','subset']:
        if cand in df.columns:
            df = df.rename(columns={cand:'split'}); break
        
    df['image_id'] = df['image_id'].astype(str).str.strip()
    df['image_id'] = df['image_id'].apply(lambda x: os.path.normpath(x) if isinstance(x,str) else x)

    df['source'] = source

    all_dfs.append(df)

merged = pd.concat(all_dfs, ignore_index=True, sort=False)
print('\nMerged rows:', len(merged))
print('Columns after merge:', merged.columns.tolist())


Reading .\fruit_quality_split_annotations.csv

Reading .\fruit_quality_adv_annotations/Annotations_ANN_pgd.csv

Reading .\fruit_quality_adv_annotations/Annotations_CNN_pgd.csv

Merged rows: 58578
Columns after merge: ['image_id', 'label', 'isNight', 'split', 'source']


5. Fix semicolon-embedded image_id rows & normalize image_path

In [20]:
def fix_semicolon_rows(df):

    df = df.copy()
    if 'image_id' not in df.columns:
        return df
    
    mask = df['image_id'].str.contains(';', na=False)
    if mask.any():
        print('Found rows where image_id contains semicolons - attempting to split into fields... Count:', mask.sum())
        parts = df.loc[mask, 'image_id'].str.split(';', expand=True)
        for i, col in enumerate(['image_id_part','label_part','isNight_part','split_part']):
            if i < parts.shape[1]:
                df.loc[mask, col] = parts[i].astype(str).str.strip()
        df.loc[mask & df['image_id_part'].notna(), 'image_id'] = df.loc[mask, 'image_id_part']

        if 'label' not in df.columns or df['label'].isna().all():
            if 'label_part' in df.columns:
                df['label'] = df['label_part']
        if 'isNight' not in df.columns or df['isNight'].isna().all():
            if 'isNight_part' in df.columns:
                df['isNight'] = df['isNight_part']
        if 'split' not in df.columns or df['split'].isna().all():
            if 'split_part' in df.columns:
                df['split'] = df['split_part']

        for c in ['image_id_part','label_part','isNight_part','split_part']:
            if c in df.columns:
                df = df.drop(columns=[c])
    else:
        print('No semicolon-embedded image_id rows found')
    return df

merged = fix_semicolon_rows(merged)

merged['image_path'] = merged['image_id'].apply(lambda x: os.path.normpath(x) if isinstance(x,str) else x)
merged[['image_id','image_path','source']].head(10)

No semicolon-embedded image_id rows found


,image_id,image_path,source
0,E:\Fruit_Quality_Split\train\bad\IMG_2510.JPG,E:\Fruit_Quality_Split\train\bad\IMG_2510.JPG,fruit_quality_split_annotations
1,E:\Fruit_Quality_Split\train\good\IMG202007281...,E:\Fruit_Quality_Split\train\good\IMG202007281...,fruit_quality_split_annotations
2,E:\Fruit_Quality_Split\train\good\IMG_9493.JPG,E:\Fruit_Quality_Split\train\good\IMG_9493.JPG,fruit_quality_split_annotations
3,E:\Fruit_Quality_Split\train\good\20190820_153...,E:\Fruit_Quality_Split\train\good\20190820_153...,fruit_quality_split_annotations
4,E:\Fruit_Quality_Split\train\good\IMG202007281...,E:\Fruit_Quality_Split\train\good\IMG202007281...,fruit_quality_split_annotations
5,E:\Fruit_Quality_Split\train\good\20190820_154...,E:\Fruit_Quality_Split\train\good\20190820_154...,fruit_quality_split_annotations
6,E:\Fruit_Quality_Split\train\good\20190820_152...,E:\Fruit_Quality_Split\train\good\20190820_152...,fruit_quality_split_annotations
7,E:\Fruit_Quality_Split\train\mixed\IMG20200729...,E:\Fruit_Quality_Split\train\mixed\IMG20200729...,fruit_quality_split_annotations
8,E:\Fruit_Quality_Split\train\good\20190820_154...,E:\Fruit_Quality_Split\train\good\20190820_154...,fruit_quality_split_annotations
9,E:\Fruit_Quality_Split\train\good\20190820_143...,E:\Fruit_Quality_Split\train\good\20190820_143...,fruit_quality_split_annotations


6. Infer adv_label from source & create stratified split

In [21]:
merged['adv_label'] = merged['source'].apply(lambda s: 0 if str(s).strip().lower() == 'fruit_quality_split_annotations' else 1)
print('adv_label distribution:')
print(merged['adv_label'].value_counts())

if 'split' not in merged.columns or merged['split'].isna().all():
    print('Creating stratified split...')
    train_frac = 0.8; val_frac=0.1; test_frac=0.1
    train, temp = train_test_split(merged, test_size=(1-train_frac), random_state=SEED, stratify=merged['adv_label'])
    val, test = train_test_split(temp, test_size=test_frac/(test_frac+val_frac), random_state=SEED, stratify=temp['adv_label'])
    train['split']='train'; val['split']='val'; test['split']='test'
    merged = pd.concat([train, val, test], ignore_index=True)
else:
    print('Using existing split')

print('\nCounts per split and adv_label:')
print(merged.groupby('split')['adv_label'].value_counts())

adv_label distribution:
adv_label
1    39052
0    19526
Name: count, dtype: int64
Using existing split

Counts per split and adv_label:
split  adv_label
test   1             3906
       0             1953
train  1            31240
       0            15620
valid  1             3906
       0             1953
Name: count, dtype: int64


7. Resolve image paths relative to DATA_ROOT and validate existence

In [22]:
def resolve_path(p):
    if p is None: return p
    p = str(p)

    if os.path.isabs(p) and os.path.exists(p):
        return p
    
    cand = os.path.join(DATA_ROOT, p)
    if os.path.exists(cand):
        return cand

    cand2 = os.path.join(DATA_ROOT, Path(p).name)
    if os.path.exists(cand2):
        return cand2
    
    cand3 = os.path.join(DATA_ROOT, 'processed_images', Path(p).name)
    if os.path.exists(cand3):
        return cand3
    
    cand4 = os.path.join(DATA_ROOT, p.replace('adv_outputs\\', '').replace('adv_outputs/', ''))
    if os.path.exists(cand4):
        return cand4

    return p

merged['image_path'] = merged['image_path'].apply(resolve_path)

exists_frac = merged['image_path'].apply(lambda x: isinstance(x,str) and os.path.exists(x)).mean()
print(f'Fraction of image paths that exist on disk: {exists_frac:.3f}')

missing = merged[~merged['image_path'].apply(lambda x: isinstance(x,str) and os.path.exists(x))]
if len(missing)>0:
    print('Sample missing (first 10):')
    display(missing[['image_id','image_path','source','adv_label']].head(10))

Fraction of image paths that exist on disk: 1.000


8. Drop helper columns and save merged annotations CSV

In [23]:
to_drop = [c for c in ['source','image_path'] if c in merged.columns]

if to_drop:
    print('Dropping columns before save:', to_drop)
    merged = merged.drop(columns=to_drop)

outpath = os.path.join(DATA_ROOT, OUT_FILENAME)

merged.to_csv(outpath, index=False)
print('Saved merged annotations to', outpath)

merged.head()

Dropping columns before save: ['source', 'image_path']
Saved merged annotations to .\fruit_quality_train_semisupervised_detector_annotations.csv


,image_id,label,isNight,split,adv_label
0,E:\Fruit_Quality_Split\train\bad\IMG_2510.JPG,1,1,train,0
1,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train,0
2,E:\Fruit_Quality_Split\train\good\IMG_9493.JPG,0,1,train,0
3,E:\Fruit_Quality_Split\train\good\20190820_153...,0,1,train,0
4,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train,0
